# Learning rate de la RNN bidireccional

¿Con qué learning rate aprende mejor `bidirectional_rnn`? Hoy usa `LEARNING_RATE = 1e-3` (el default de Adam).

El learning rate es el tamaño de cada paso de AdamW:

- **muy chico**: la loss baja lento y en pocas épocas la red queda sub-entrenada (la mejor época es la última);
- **muy grande**: los pasos se pasan del mínimo, la loss oscila o sube y el F1 se estanca;
- **bueno**: la loss de test baja rápido y se estabiliza.

Por eso además del F1 importan las **curvas por época**: muestran si a un `lr` le faltaron épocas o si es inestable.

Cada problema entrena con el weight decay que ya eligió (`bidirectional_rnn_weight_decay`: 0 en
capitalización, 1 en puntuación) y sin dropout. Ojo: AdamW achica los pesos `lr · λ` por paso, así que en
puntuación cambiar el `lr` también cambia cuánto regulariza el decay.

**Corre local, en CPU**: con 300 instancias por fuente y 15 épocas tarda ~12 minutos (5 learning rates × 2
problemas). Son menos datos que el pipeline (1000), así que el `lr` elegido acá hay que confirmarlo con
el pipeline completo antes de fijarlo.

## 1. Setup

In [ ]:
import json
import logging
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any

# El notebook corre desde notebooks/experiments/: subimos hasta la raíz del repo para importar src
while not Path("pyproject.toml").exists():
    os.chdir("..")

import matplotlib.pyplot as plt
import torch
from IPython.display import display
from torch.utils.data import DataLoader

from notebooks.experiments.regularization.regularization_plots import (
    best_value,
    plot_control_f1,
    plot_gap,
    plot_loss_curves,
    summary_table,
)
from src.constants import MODELS_DIR, RUN_SEPARATOR
from src.data import build_loader, preprocess_dataset, split_dataset_dev_control, split_train_test
from src.features import EmbeddingsExtractor
from src.metrics.base import scores_from_tasks
from src.problem import CapitalizationProblem, Problem, PunctuationProblem
from src.training.neural import evaluate_problem_network, train_neural_model
from src.training.result import format_score

%matplotlib inline
# Solo warnings de src: el progreso lo imprime el notebook, una línea por entrenamiento
logging.basicConfig(level=logging.WARNING)
torch.set_num_threads(os.cpu_count() or 1)
print("Directorio:", Path.cwd(), "| GPU:", torch.cuda.is_available())

## 2. Configuración

In [ ]:
PARAM = "learning_rate"
# Escala logarítmica alrededor del actual (1e-3): el efecto del lr es multiplicativo
LEARNING_RATES = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]

EPOCHS = 15
# Instancias por fuente. 300 ≈ 4.5 s por época en CPU; el pipeline usa 1000 (~3 veces más lento)
CAPITALIZATION_SIZE = 300
PUNCTUATION_SIZE = 300

PROBLEMS = [CapitalizationProblem(), PunctuationProblem()]
print(f"{len(LEARNING_RATES) * len(PROBLEMS)} entrenamientos de {EPOCHS} épocas")

## 3. Datos

Mismas fuentes, ventanas, splits y semilla que el pipeline: desarrollo/control y, dentro de desarrollo,
train/test. Se generan una sola vez y se reusan para todos los learning rates.

In [ ]:
def problem_loaders(dataset) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Train, test y control con los mismos splits que el pipeline"""
    development_set, control_set = split_dataset_dev_control(dataset)
    train_set, test_set = split_train_test(development_set)
    print(f"  train={len(train_set)} test={len(test_set)} control={len(control_set)}")
    return (
        build_loader(train_set, shuffle=True),
        build_loader(test_set, shuffle=False),
        build_loader(control_set, shuffle=False),
    )


punctuation_dataset, capitalization_dataset = preprocess_dataset(PUNCTUATION_SIZE, CAPITALIZATION_SIZE)
datasets = {"CapitalizationProblem": capitalization_dataset, "PunctuationProblem": punctuation_dataset}
loaders = {}
for problem in PROBLEMS:
    name = type(problem).__name__
    print(name)
    loaders[name] = problem_loaders(datasets[name])
extractor = EmbeddingsExtractor()

## 4. Barrido

In [ ]:
def history_rows(result) -> list[dict[str, Any]]:
    """Loss y F1 de train y test de cada época"""
    return [
        {"epoch": m.epoch, "train_loss": m.loss, "test_loss": m.test_loss, "train_f1": m.score, "test_f1": m.test_score}
        for m in result.history
    ]


def train_with_lr(problem: Problem, learning_rate: float) -> dict[str, Any]:
    """Entrena la bidireccional del problema con ese lr y devuelve sus métricas"""
    train_loader, test_loader, control_loader = loaders[type(problem).__name__]
    model = problem.bidirectional_rnn_model()
    result = train_neural_model(
        train_loader, test_loader, extractor, problem.metric, model, n_epochs=EPOCHS,
        weight_decay=problem.bidirectional_rnn_weight_decay, learning_rate=learning_rate,
    )
    control = scores_from_tasks(evaluate_problem_network(result.model, control_loader, extractor, problem))
    return {"value": learning_rate, "best_epoch": result.best_epoch, "test_f1": result.score,
            "control_f1": control, "history": history_rows(result)}


stamp = datetime.now().strftime(f"%Y-%m-%d{RUN_SEPARATOR}%H-%M-%S")
OUT_DIR = Path(MODELS_DIR) / "learning_rate" / stamp
OUT_DIR.mkdir(parents=True, exist_ok=True)
report = {"config": {"learning_rates": LEARNING_RATES, "epochs": EPOCHS, "capitalization_size": CAPITALIZATION_SIZE,
                     "punctuation_size": PUNCTUATION_SIZE}, "problems": {}}

for problem in PROBLEMS:
    runs = report["problems"].setdefault(type(problem).__name__, [])
    for lr in LEARNING_RATES:
        start = time.time()
        runs.append(train_with_lr(problem, lr))
        run = runs[-1]
        print(f"{type(problem).__name__:22} lr={lr:<7g} mejor época={run['best_epoch']:>2}  "
              f"test: {format_score(run['test_f1'])}  ({time.time() - start:.0f} s)")
        (OUT_DIR / "results.json").write_text(json.dumps(report, indent=2))  # por si se corta
print("Resultados en", OUT_DIR)

## 5. Resultados

### Tabla

- **época**: la mejor en test. Si es la última (15), a ese `lr` le faltaron épocas.
- **brecha**: F1 de train − F1 de test en esa época.
- **test / control**: F1 macro promedio. Se elige con test; control confirma.

In [ ]:
def show_and_save(fig, name: str) -> None:
    """Muestra el gráfico y lo guarda como PNG junto al results.json"""
    fig.savefig(OUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)


print(summary_table(report))

### Curvas de loss

Lo más informativo para el learning rate: con `lr` chico la loss baja lento y sin estabilizarse, con
`lr` grande baja rápido pero la de test (llena) se pone ruidosa o sube.

In [ ]:
show_and_save(plot_loss_curves(report, PARAM), "loss_learning_rate")

### F1 de test por época

In [ ]:
def plot_test_f1_curves(report: dict[str, Any]) -> plt.Figure:
    """F1 promedio de test en cada época, una curva por lr: qué tan rápido y estable aprende"""
    problems = report["problems"]
    fig, axes = plt.subplots(1, len(problems), figsize=(6 * len(problems), 4), squeeze=False)
    for ax, (problem, runs) in zip(axes[0], problems.items()):
        for run in runs:
            f1 = [sum(h["test_f1"].values()) / len(h["test_f1"]) for h in run["history"]]
            ax.plot(range(1, len(f1) + 1), f1, marker=".", label=f"{run['value']:g}")
        ax.set(title=problem, xlabel="época", ylabel="F1 macro promedio (test)")
        ax.grid(alpha=0.3)
        ax.legend(title="lr")
    fig.tight_layout()
    return fig


show_and_save(plot_test_f1_curves(report), "f1_test_por_epoca")

### F1 en control y sobreajuste

In [ ]:
show_and_save(plot_control_f1(report, PARAM), "f1_control_learning_rate")
show_and_save(plot_gap(report, PARAM), "brecha_learning_rate")

## 6. Mejor learning rate por problema

In [ ]:
for problem in report["problems"]:
    print(f"{problem}: lr = {best_value(report, problem):g} (test), {best_value(report, problem, 'control'):g} (control)")

## 7. Conclusión

_Completar después de correr_: qué `lr` conviene y por qué (F1 en test confirmado en control, mejor época
y forma de las curvas). Si el mejor es un extremo de la grilla, ampliarla hacia ese lado. Antes de fijarlo en
el código, confirmarlo con el pipeline completo (1000 instancias por fuente).